# TradeFlowGNN: Model Evaluation

This notebook loads the trained checkpoints and compares the GCN performance against baselines.

In [ ]:
# ── Colab/Notebook Setup ───────────────────────────────────────────
import sys
from pathlib import Path
import os

def setup_colab():
    if 'google.colab' in str(get_ipython()):
        print("Running in Google Colab. Setting up environment...")
        
        # 1. Install dependencies
        !pip install -q torch-geometric pytorch-lightning pyarrow pandas numpy matplotlib seaborn tqdm pyyaml
        
        # 2. Clone repository if not already present
        repo_name = "TradeFlowGCN"
        if not Path(repo_name).exists() and not Path("src").exists():
            !git clone https://github.com/tolyaho/TradeFlowGCN.git
        
        # 3. Enter repository directory
        if Path(repo_name).exists() and not Path("src").exists():
            os.chdir(repo_name)
            print(f"Changed directory to {os.getcwd()}")
        
        # 4. Set PYTHONPATH
        if os.getcwd() not in sys.path: sys.path.append(os.getcwd())
        src_path = os.path.join(os.getcwd(), "src")
        if src_path not in sys.path: sys.path.append(src_path)

setup_colab()

# ── Robust Path Setup ──────────────────────────────────────────────
curr_path = Path(".").resolve()
root_path = curr_path
while not (root_path / "src").exists() and root_path.parent != root_path:
    root_path = root_path.parent

if (root_path / "src").exists():
    if str(root_path / "src") not in sys.path:
        sys.path.append(str(root_path / "src"))
    print(f"Project Root: {root_path}")
else:
    print(f"Warning: Could not find 'src' directory relative to {curr_path}")

## 0. Training (Optional)

If you are in a fresh Colab environment, your `lightning_logs` will be empty. Run the cell below to train the model for a few epochs to generate a checkpoint.

In [12]:
# Run training script directly from the notebook
!python scripts/download_data.py
!python scripts/train.py --config configs/default.yaml

2026-03-24 04:55:13,702 | INFO | trade_flow_gcn.data.download | Downloading CEPII Gravity data from http://www.cepii.fr/DATA_DOWNLOAD/gravity/data/Gravity_csv_V202211.zip ...
2026-03-24 04:55:15,983 | INFO | trade_flow_gcn.data.download |   Downloaded 1.0 MB ...
2026-03-24 04:55:16,258 | INFO | trade_flow_gcn.data.download |   Downloaded 2.1 MB ...
2026-03-24 04:55:16,509 | INFO | trade_flow_gcn.data.download |   Downloaded 3.1 MB ...
2026-03-24 04:55:16,512 | INFO | trade_flow_gcn.data.download |   Downloaded 4.2 MB ...
2026-03-24 04:55:16,651 | INFO | trade_flow_gcn.data.download |   Downloaded 5.2 MB ...
2026-03-24 04:55:16,671 | INFO | trade_flow_gcn.data.download |   Downloaded 6.3 MB ...
2026-03-24 04:55:16,792 | INFO | trade_flow_gcn.data.download |   Downloaded 7.3 MB ...
2026-03-24 04:55:16,807 | INFO | trade_flow_gcn.data.download |   Downloaded 8.4 MB ...
2026-03-24 04:55:16,928 | INFO | trade_flow_gcn.data.download |   Downloaded 9.4 MB ...
2026-03-24 04:55:16,937 | INFO | 

## 1. Find Checkpoints

Lightning saves checkpoints in `lightning_logs/` by default.

In [ ]:
log_dir = root_path / "lightning_logs"
ckpt_files = list(log_dir.glob("**/checkpoints/*.ckpt"))

if not ckpt_files:
    print(f"No checkpoints found in {log_dir}. Please run the training cell above first.")
else:
    # Get the latest checkpoint based on file modification time
    latest_ckpt = sorted(ckpt_files, key=lambda p: p.stat().st_mtime)[-1]
    print(f"Latest checkpoint: {latest_ckpt}")

## 2. Load Model

In [ ]:
from trade_flow_gcn.models.gcn import TradeFlowGCN
from trade_flow_gcn.training.lightning_module import TradeFlowModule
from trade_flow_gcn.utils.config import load_config

# Load config
config = load_config(str(root_path / "configs/default.yaml"))

# Initialize model architecture
model = TradeFlowGCN(
    node_in_channels=len(config['data']['node_features']),
    edge_in_channels=len(config['data']['edge_features']),
    hidden_channels=64,
    out_channels=1
)

if 'latest_ckpt' in locals():
    # Load weights
    module = TradeFlowModule.load_from_checkpoint(latest_ckpt, model=model)
    module.eval()
    print("Model loaded successfully.")
else:
    print("Cannot load model: No checkpoint found.")